# 03 · Experiments & Serving

(1) Per-aspect **ensemble**; (2) the **novel experiment** — one shared model vs 10
per-queue specialists; (3) live serving demo.

## Setup

In [1]:
import os, sys, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = os.path.abspath('..'); sys.path.insert(0, ROOT)
from taxonomy import ASPECTS, ASPECT_DEFINITIONS, QUEUES, SENTIMENT_LABELS, ASPECT_MAP, map_aspects
DATA = os.path.join(ROOT, 'data')

In [2]:
import joblib, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tr=pd.read_csv(f'{DATA}/train.csv'); va=pd.read_csv(f'{DATA}/val.csv'); te=pd.read_csv(f'{DATA}/test.csv')
L2={'not_present':0,'negative':1,'neutral':2}; HYP={a:f'{a}: {ASPECT_DEFINITIONS[a]}' for a in ASPECTS}

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


## 1 · Per-aspect ensemble

For each aspect, pick the model that wins on **val** (never peeks at test), then score
that routed combination on test.

In [3]:
VEC=joblib.load(f'{ROOT}/models/tfidf/vectorizer.joblib')
Xva,Xte=VEC.transform(va.body.fillna('')), VEC.transform(te.body.fillna(''))
dev='cuda' if torch.cuda.is_available() else 'cpu'
tok=AutoTokenizer.from_pretrained(f'{ROOT}/models/roberta-absa')
model=AutoModelForSequenceClassification.from_pretrained(f'{ROOT}/models/roberta-absa').to(dev).eval()
def rb(df,a):
    out=[]; b=df.body.fillna('').tolist()
    for i in range(0,len(b),64):
        bb=b[i:i+64]
        e=tok(bb,[HYP[a]]*len(bb),truncation=True,max_length=256,padding=True,return_tensors='pt').to(dev)
        with torch.no_grad(): out.extend(model(**e).logits.argmax(-1).cpu().numpy().tolist())
    return np.array(out)
rows=[]; ens=[]
for a in ASPECTS:
    clf=joblib.load(f'{ROOT}/models/tfidf/aspect_{a}.joblib')
    gv=va[f'asp_{a}'].fillna('not_present').replace('','not_present')
    f_tf=f1_score(gv.map(L2), pd.Series(clf.predict(Xva)).map(L2), average='macro', zero_division=0)
    f_rb=f1_score(gv.map(L2), rb(va,a), average='macro', zero_division=0)
    pick='roberta' if f_rb>=f_tf else 'tfidf'
    gt=te[f'asp_{a}'].fillna('not_present').replace('','not_present').map(L2).values
    tt=f1_score(gt, pd.Series(clf.predict(Xte)).map(L2), average='macro', zero_division=0)
    rt=f1_score(gt, rb(te,a), average='macro', zero_division=0)
    et=rt if pick=='roberta' else tt; ens.append(et)
    rows.append({'aspect':a,'pick':pick,'tfidf':round(tt,3),'roberta':round(rt,3),'ensemble':round(et,3)})
display(pd.DataFrame(rows))
print('ENSEMBLE mean aspect macro-F1 (test): %.3f' % np.mean(ens))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,aspect,pick,tfidf,roberta,ensemble
0,billing,roberta,0.692,0.686,0.686
1,account,roberta,0.765,0.798,0.798
2,performance,roberta,0.813,0.840,0.840
3,security,roberta,0.704,0.743,0.743
4,bug,tfidf,0.633,0.587,0.633
5,feature,roberta,0.743,0.772,0.772
6,network,roberta,0.709,0.692,0.692
7,hardware,tfidf,0.657,0.667,0.657
8,software,roberta,0.803,0.838,0.838


ENSEMBLE mean aspect macro-F1 (test): 0.740


## 2 · Novel experiment — shared model vs per-queue specialists

Does **one shared model** match **N per-queue specialists**? TF-IDF for both (feasible
on small queues a transformer can't fit). Per-queue mean aspect macro-F1 on test.

In [4]:
def fit(slice_):
    v=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=30000,sublinear_tf=True,strip_accents='unicode')
    X=v.fit_transform(slice_.body.fillna('')); clfs={}
    for a in ASPECTS:
        y=slice_[f'asp_{a}'].fillna('not_present').replace('','not_present')
        clfs[a]=('const',y.iloc[0]) if y.nunique()<2 else ('m',LogisticRegression(max_iter=2000,class_weight='balanced',C=2.0).fit(X,y))
    return v,clfs
def score(v,clfs,sl):
    if len(sl)==0: return np.nan
    X=v.transform(sl.body.fillna('')); fs=[]
    for a in ASPECTS:
        y=sl[f'asp_{a}'].fillna('not_present').replace('','not_present'); k,o=clfs[a]
        pred=np.array([o]*len(sl)) if k=='const' else o.predict(X)
        fs.append(f1_score(y,pred,average='macro',zero_division=0))
    return np.mean(fs)
vs,cs=fit(tr); rows=[]
for q in QUEUES:
    trq,teq=tr[tr.queue==q],te[te.queue==q]
    sh=score(vs,cs,teq)
    if len(trq)>=30: vq,cq=fit(trq); sp=score(vq,cq,teq)
    else: sp=np.nan
    rows.append({'queue':q,'n_te':len(teq),'shared':round(sh,3),
                 'per_queue':round(sp,3) if sp==sp else None,
                 'delta':round(sp-sh,3) if sp==sp else None})
R=pd.DataFrame(rows); display(R)
v=R.dropna(subset=['per_queue']); w=v.n_te
print('SHARED   (weighted): %.3f' % np.average(v.shared,weights=w))
print('PER-QUEUE(weighted): %.3f' % np.average(v.per_queue,weights=w))
print('models: 1 shared  vs  %d per-queue' % len(v))

,queue,n_te,shared,per_queue,delta
0,Technical Support,884,0.705,0.667,-0.038
1,Product Support,539,0.710,0.664,-0.045
2,Customer Service,412,0.706,0.679,-0.027
3,IT Support,359,0.734,0.703,-0.031
4,Billing and Payments,339,0.673,0.660,-0.013
5,Returns and Exchanges,142,0.707,0.680,-0.027
6,Service Outages and Maintenance,121,0.802,0.758,-0.044
7,Sales and Pre-Sales,86,0.772,0.652,-0.121
8,Human Resources,65,0.760,0.686,-0.074
9,General Inquiry,53,0.693,0.655,-0.038


SHARED   (weighted): 0.713
PER-QUEUE(weighted): 0.676
models: 1 shared  vs  10 per-queue


**Finding:** the single shared model beats per-queue specialists on **10/10**
queues (+0.037 weighted) at **1/10** the maintenance. Specialization fragments scarce
data and loses cross-aspect transfer — the project's headline result.

## 3 · Serving demo

`app/predict.py` is the inference core the FastAPI service (`app/api.py`) exposes; the
Streamlit UI (`app/streamlit_app.py`) calls that API over HTTP.

In [5]:
from app.predict import predict
predict('I was overcharged on my invoice and the app keeps crashing during checkout.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'aspects': {'billing': 'negative',
  'account': 'not_present',
  'performance': 'not_present',
  'security': 'not_present',
  'bug': 'negative',
  'feature': 'not_present',
  'network': 'not_present',
  'hardware': 'not_present',
  'software': 'negative'},
 'present': {'billing': 'negative', 'bug': 'negative', 'software': 'negative'},
 'negatives': ['billing', 'bug', 'software'],
 'route': 'Billing and Payments',
 'priority': 'high'}